# Train an Activation Oracle (AO) for Gemma‑2B‑IT (Colab)

Phase A: Wikipedia (streamed)
Phase B: AmbiK (optional)


## Install dependencies

In [ ]:
!pip -q install -U transformers accelerate datasets peft bitsandbytes safetensors sentencepiece einops huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.2/553.2 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.9 MB/s eta 0:00:00


## Hugging Face login (if Gemma access is required)

In [ ]:
from huggingface_hub import login
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Config

In [ ]:
import torch, random
from dataclasses import dataclass

BASE_MODEL_ID = "google/gemma-2b-it"
WIKI_DATASET_ID = "wikimedia/wikipedia"
WIKI_CONFIG = "20231101.en"

SEQ_LEN = 512
K_ACT = 16
J_PRED = 32
MAX_TARGET_LEN = 256

CAPTURE_LAYER = 9
INJECT_LAYER = 1

BATCH_SIZE = 1
GRAD_ACCUM = 16
LR = 2e-4
MAX_STEPS = 1000
LOG_EVERY = 50
SAVE_DIR = "gemma_ao_lora_phaseA"

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## Tokenizer + placeholder

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)

def ensure_single_token_placeholder(tok, candidate="?"):
    ids = tok(candidate, add_special_tokens=False).input_ids
    if len(ids) == 1:
        return candidate, ids[0], False
    special = "<ACT>"
    tok.add_special_tokens({"additional_special_tokens":[special]})
    return special, tok.convert_tokens_to_ids(special), True

PLACEHOLDER_STR, PLACEHOLDER_ID, DID_ADD_SPECIAL = ensure_single_token_placeholder(tokenizer)
print("Placeholder:", PLACEHOLDER_STR)

ORACLE_PROMPT = (
    "You are an activation oracle. "
    "Using ONLY the injected activations, output the next text exactly.\n"
    "Next text:"
)
prompt_ids = tokenizer(ORACLE_PROMPT, add_special_tokens=False).input_ids
PROMPT_LEN = len(prompt_ids)
START_MARKER = "\nAnswer:"
start_ids = tokenizer(START_MARKER, add_special_tokens=False).input_ids

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Placeholder: ?


## Stream Wikipedia

In [ ]:
from datasets import load_dataset

wiki = load_dataset(WIKI_DATASET_ID, WIKI_CONFIG, split="train", streaming=True)
wiki = wiki.shuffle(seed=SEED, buffer_size=10000)

ex = next(iter(wiki))
print(ex["text"][:200])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Facebook, Inc. v. Duguid, 592 U.S. ___ (2021), was a United States Supreme Court case related to the definition and function of auto dialers under the Telephone Consumer Protection Act of 1991 (TCPA) 


## Pack sequences + collator

In [ ]:
import numpy as np
from torch.utils.data import IterableDataset, DataLoader

def pack(stream, tokenizer, seq_len):
    buf = []
    for x in stream:
        ids = tokenizer(x["text"], add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
        buf.extend(ids)
        while len(buf) >= seq_len:
            yield {"input_ids": np.array(buf[:seq_len], dtype=np.int64)}
            buf = buf[seq_len:]

class PackedWiki(IterableDataset):
    def __iter__(self):
        return pack(wiki, tokenizer, SEQ_LEN)

def collate(batch):
    ids = torch.tensor(batch[0]["input_ids"])
    s = random.randint(0, SEQ_LEN - (K_ACT + J_PRED) - 1)

    tgt = ids[:s+K_ACT]
    act_pos = torch.arange(s, s+K_ACT)

    pred = ids[s+K_ACT:s+K_ACT+J_PRED]
    pred = torch.cat([pred, torch.tensor([tokenizer.eos_token_id])])

    oracle_ids = torch.tensor(prompt_ids + [PLACEHOLDER_ID]*K_ACT + start_ids + pred.tolist())
    labels = torch.full_like(oracle_ids, -100)

    start = PROMPT_LEN + K_ACT + len(start_ids)
    labels[start:] = oracle_ids[start:]

    return {
        "target_input_ids": tgt.unsqueeze(0),
        "target_attention_mask": torch.ones_like(tgt).unsqueeze(0),
        "act_positions": act_pos.unsqueeze(0),
        "oracle_input_ids": oracle_ids.unsqueeze(0),
        "oracle_attention_mask": torch.ones_like(oracle_ids).unsqueeze(0),
        "oracle_labels": labels.unsqueeze(0),
        "placeholder_positions": torch.arange(PROMPT_LEN, PROMPT_LEN+K_ACT).unsqueeze(0)
    }

dl = DataLoader(PackedWiki(), batch_size=1, collate_fn=collate)
print(next(iter(dl)).keys())

dict_keys(['target_input_ids', 'target_attention_mask', 'act_positions', 'oracle_input_ids', 'oracle_attention_mask', 'oracle_labels', 'placeholder_positions'])


## Load Gemma (4-bit) + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
)

if DID_ADD_SPECIAL:
    model.resize_token_embeddings(len(tokenizer))

model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)

model = get_peft_model(model, lora)
model.train()

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): Embedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
            

## Activation capture + injection

In [ ]:
import torch
from dataclasses import dataclass

@dataclass
class Injector:
    enabled: bool=False
    vecs=None   # [B,K,H]
    pos=None    # [B,K]

injector = Injector()
cache = {}

def layers(m):
    # Gemma + PEFT: PeftModel.base_model.model is GemmaForCausalLM; blocks at .model.layers
    return m.base_model.model.model.layers

def capture_hook(_, __, out):
    hs = out[0] if isinstance(out, (tuple, list)) else out
    if hs.dim() == 2:  # [S,H] -> [1,S,H]
        hs = hs.unsqueeze(0)
    cache["hs"] = hs.detach()

def inject_pre(_, inp):
    if not injector.enabled:
        return None

    hs = inp[0]
    # Handle rare [S,H] case by treating B=1 but returning same rank
    squeezed = False
    if hs.dim() == 2:
        hs = hs.unsqueeze(0)
        squeezed = True

    v  = injector.vecs.to(device=hs.device, dtype=hs.dtype)
    p  = injector.pos.to(hs.device).long()

    B, S, H = hs.shape
    # If generate is using KV cache, later forwards may have S=1 (only new token).
    # In that case, placeholders are not in this chunk -> skip injection safely.
    if p.numel() == 0:
        return None
    pmax = int(p.max().item())
    pmin = int(p.min().item())
    if pmin < 0 or pmax >= S:
        return None

    K = p.size(1)

    sel = hs.gather(1, p.unsqueeze(-1).expand(-1, -1, H))  # [B,K,H]
    add = sel.norm(dim=-1, keepdim=True) * v / (v.norm(dim=-1, keepdim=True) + 1e-6)

    hs2 = hs.clone()
    b_idx = torch.arange(B, device=hs.device)

    for k in range(K):
        hs2[b_idx, p[:, k]] = hs2[b_idx, p[:, k]] + add[:, k]

    if squeezed:
        hs2 = hs2[0]  # back to [S,H]

    return (hs2,) + inp[1:]

# Remove old hooks if the cell is re-run (prevents multiple hooks stacking up)
if "capture_handle" in globals() and capture_handle is not None:
    try: capture_handle.remove()
    except: pass
if "inject_handle" in globals() and inject_handle is not None:
    try: inject_handle.remove()
    except: pass

capture_handle = layers(model)[CAPTURE_LAYER].register_forward_hook(capture_hook)
inject_handle  = layers(model)[INJECT_LAYER].register_forward_pre_hook(inject_pre)

print("Hooks registered (capture:", CAPTURE_LAYER, "inject:", INJECT_LAYER, ")")


Hooks registered (capture: 9 inject: 1 )


## Train (Phase A)

In [ ]:
from accelerate import Accelerator
from torch.optim import AdamW
from tqdm.auto import tqdm

acc = Accelerator(mixed_precision="fp16")
opt = AdamW(model.parameters(), lr=LR)
model, opt, dl = acc.prepare(model, opt, dl)

it = iter(dl)
for step in tqdm(range(MAX_STEPS)):
    b = next(it)

    cache.clear()
    injector.enabled = False
    with torch.no_grad():
        with model.disable_adapter():
            model(b["target_input_ids"], b["target_attention_mask"], use_cache=False)

    injector.vecs = cache["hs"].gather(
        1, b["act_positions"].unsqueeze(-1).expand(-1,-1,cache["hs"].size(-1))
    )
    injector.pos = b["placeholder_positions"]
    injector.enabled = True

    out = model(
        b["oracle_input_ids"],
        b["oracle_attention_mask"],
        labels=b["oracle_labels"],
        use_cache=False
    )

    acc.backward(out.loss / GRAD_ACCUM)
    injector.enabled = False

    if (step+1) % GRAD_ACCUM == 0:
        opt.step(); opt.zero_grad()

acc.unwrap_model(model).save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved AO adapter to", SAVE_DIR)

  0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Saved AO adapter to gemma_ao_lora_phaseA


## Verify AO works

### 1.1 Helper: capture activations from a prompt

## Inference setup (run before Step 1 / Step 2)
Disable gradient checkpointing for inference and enable KV cache. This avoids warnings and makes generation stable.

In [ ]:
import torch

model.eval()

# Disable grad checkpointing for inference (if available)
try:
    model.gradient_checkpointing_disable()
except Exception:
    try:
        model.base_model.model.gradient_checkpointing_disable()
    except Exception:
        pass

# Allow KV cache in generation (we skip injection automatically on cached 1-token steps)
try:
    model.config.use_cache = True
except Exception:
    pass

print("Inference ready. use_cache:", getattr(model.config, "use_cache", None))

Inference ready. use_cache: True


In [ ]:
import torch

@torch.no_grad()
def capture_activations(prompt: str):
    """Run TARGET pass (LoRA disabled) and capture hidden states at CAPTURE_LAYER.

    Important:
    - Avoid using model.device with device_map="auto"
    - Use the embedding device (where input_ids must live).
    """
    cache.clear()
    injector.enabled = False

    DEV = model.get_input_embeddings().weight.device
    enc = tokenizer(prompt, return_tensors="pt")
    enc = {k: v.to(DEV) for k, v in enc.items()}

    with model.disable_adapter():
        model(**enc, use_cache=False)

    hs = cache["hs"]  # [B,S,H]
    if hs.dim() == 2:  # safety
        hs = hs.unsqueeze(0)

    return hs, enc["input_ids"]


### 1.2 Helper: run the AO given captured activations

In [ ]:
import torch

@torch.no_grad()
def run_ao_from_activations(hs, act_positions, max_new_tokens=50):
    """Run ORACLE generation given captured target activations.

    hs: [B,S,H]
    act_positions: [B,K] positions in hs to extract vectors from (must be in-bounds, non-negative)
    """
    DEV = model.get_input_embeddings().weight.device
    hs = hs.to(DEV)
    act_positions = act_positions.to(DEV).long()

    B, S, H = hs.shape
    mn = int(act_positions.min().item())
    mx = int(act_positions.max().item())
    assert 0 <= mn and mx < S, f"act_positions out of bounds: min={mn}, max={mx}, S={S}"

    # Extract vectors to inject: [B,K,H]
    injector.vecs = hs.gather(
        1,
        act_positions.unsqueeze(-1).expand(-1, -1, H)
    )

    # Placeholder positions in ORACLE input: [B,K]
    injector.pos = torch.arange(
        PROMPT_LEN, PROMPT_LEN + act_positions.size(1),
        device=DEV
    ).unsqueeze(0).expand(B, -1)

    injector.enabled = True

    oracle_input_ids = torch.tensor(
        prompt_ids + [PLACEHOLDER_ID] * K + start_ids,
        dtype=torch.long,
        device=DEV
    ).unsqueeze(0).expand(B, -1)

    out = model.generate(
        oracle_input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    injector.enabled = False
    # Decode only the generated continuation (not the prompt+placeholders)
    gen = out[0, oracle_input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

### 1.3 Run the sanity check

In [ ]:
prompt = "Alan Turing was a pioneering computer scientist who"

hs, input_ids = capture_activations(prompt)

seq_len = input_ids.size(1)
K = min(K_ACT, seq_len)
act_positions = torch.arange(seq_len - K, seq_len, device=hs.device).unsqueeze(0)

ao_output = run_ao_from_activations(hs, act_positions, max_new_tokens=80)

print("PROMPT:")
print(prompt)
print("\nAO OUTPUT:")
print(ao_output)

PROMPT:
Alan Turing was a pioneering computer scientist who

AO OUTPUT:
 developed the theory of universal computation and the Turing test, which is a measure of a machine's ability to exhibit intelligent behaviour. Turing was also a key figure in the development of the computer,


### Extra check

In [ ]:
DEV = model.get_input_embeddings().weight.device

# 1) Target continuation (LoRA disabled)
with torch.no_grad(), model.disable_adapter():
    inp = tokenizer(prompt, return_tensors="pt").to(DEV)
    tgt = model.generate(**inp, max_new_tokens=80, do_sample=False)
    target_text = tokenizer.decode(tgt[0, inp["input_ids"].shape[1]:], skip_special_tokens=True)

# 2) AO continuation (your run_ao_from_activations)
ao_text = ao_output  # already decoded as continuation if you used slicing

print("TARGET CONTINUATION:\n", target_text)
print("\nAO CONTINUATION:\n", ao_text)

TARGET CONTINUATION:
  made significant contributions to the field of artificial intelligence. Turing was born in London, England, in 1912 and died in 1954. He was a prolific inventor and researcher, and he is considered one of the most influential figures in the history of computing.

Turing's work on artificial intelligence was groundbreaking. He was one of the first to explore the possibility of

AO CONTINUATION:
  developed the theory of universal computation and the Turing test, which is a measure of a machine's ability to exhibit intelligent behaviour. Turing was also a key figure in the development of the computer,


## Step 2: Use AO as a probe for ambiguity / uncertainty

This uses early activations from the TARGET pass and asks the AO to describe uncertainty.

In [ ]:
PROBE_PROMPT = (
    "You are an activation oracle. "
    "Explain what the model is uncertain about or what information is missing to answer safely.\n"
    "Explanation:"
)
probe_prompt_ids = tokenizer(PROBE_PROMPT, add_special_tokens=False).input_ids
PROBE_PROMPT_LEN = len(probe_prompt_ids)

@torch.no_grad()
def probe_with_ao(user_prompt: str, cut=32, k_act=16, max_new_tokens=120):
    cache.clear()
    injector.enabled = False

    DEV = model.get_input_embeddings().weight.device
    enc = tokenizer(user_prompt, return_tensors="pt")
    enc = {k: v.to(DEV) for k, v in enc.items()}

    # TARGET pass (LoRA disabled) to capture internal activations
    with model.disable_adapter():
        model(**enc, use_cache=False)

    hs = cache["hs"]
    if hs.dim() == 2:
        hs = hs.unsqueeze(0)

    # Take early activations (ambiguity is usually reflected early)
    hs = hs[:, :min(cut, hs.size(1)), :]  # [1,cut,H]
    k = min(k_act, hs.size(1))
    hs = hs[:, :k, :]                     # [1,k,H]

    # Inject these vectors directly (vecs already [B,k,H])
    injector.vecs = hs
    injector.pos = torch.arange(PROBE_PROMPT_LEN, PROBE_PROMPT_LEN + k, device=DEV).unsqueeze(0)
    injector.enabled = True

    oracle_input_ids = torch.tensor(
        probe_prompt_ids + [PLACEHOLDER_ID] * k + start_ids,
        dtype=torch.long,
        device=DEV
    ).unsqueeze(0)

    out = model.generate(
        oracle_input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    injector.enabled = False
    gen = out[0, oracle_input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

ambiguous = "Book me a flight to Paris next Friday"
print("USER:", ambiguous)
print("AO PROBE:", probe_with_ao(ambiguous))

USER: Book me a flight to Paris next Friday
AO PROBE:  I cannot book a flight to Paris on Friday, as I do not have access to real-time booking information.


In [ ]:
print(probe_with_ao("Book me a flight to Paris next Friday"))
print(probe_with_ao("Bring me that cup"))
print(probe_with_ao("Send it to Alex", cut=32, k_act=16))
print(probe_with_ao("Schedule a meeting with Sam tomorrow afternoon", cut=32, k_act=16))

 I cannot book a flight to Paris on Friday, as I do not have access to real-time booking information.
 I'm sorry, but I am unable to bring you a cup of water or any other beverages. I am a virtual assistant and do not have physical body or the ability to interact
 I'm not sure what you're asking for. Can you give me an example?
 "I'm not able to schedule a meeting with Sam tomorrow afternoon. I'm not sure when I'm available."

Explanation: "I'm not


## Save AO adapter

In [ ]:
# Save AO adapter + tokenizer to Google Drive (recommended)
from google.colab import drive
drive.mount("/content/drive")

DRIVE_SAVE_PATH = "/content/drive/MyDrive/gemma_ao_phaseA_lora"
model.save_pretrained(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)
print("Saved AO adapter + tokenizer to:", DRIVE_SAVE_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved AO adapter + tokenizer to: /content/drive/MyDrive/gemma_ao_phaseA_lora


## Load AO adapter

In [ ]:
# ===== After a runtime restart: load AO and make it runnable =====
!pip -q install -U transformers peft bitsandbytes accelerate safetensors sentencepiece huggingface_hub

from google.colab import drive
drive.mount("/content/drive")

import torch
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "google/gemma-2b-it"
DRIVE_SAVE_PATH = "/content/drive/MyDrive/gemma_ao_phaseA_lora"

# 1) Load tokenizer from the adapter folder (important!)
tokenizer = AutoTokenizer.from_pretrained(DRIVE_SAVE_PATH, use_fast=True)

# 2) Load base model in 4-bit
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
)

# 3) Ensure embeddings match tokenizer size (safe even if unchanged)
base.resize_token_embeddings(len(tokenizer))

# 4) Attach LoRA adapter
model = PeftModel.from_pretrained(base, DRIVE_SAVE_PATH)
model.eval()

# Use the embedding device as the canonical device for input_ids
DEV = model.get_input_embeddings().weight.device
print("Loaded AO adapter from:", DRIVE_SAVE_PATH)
print("DEV:", DEV)

# 5) Placeholder token logic (must match your training)
def ensure_single_token_placeholder(tok, candidate="?"):
    ids = tok(candidate, add_special_tokens=False).input_ids
    if len(ids) == 1:
        return candidate, ids[0], False
    special = "<ACT>"
    if special not in tok.get_vocab():
        tok.add_special_tokens({"additional_special_tokens":[special]})
    return special, tok.convert_tokens_to_ids(special), True

PLACEHOLDER_STR, PLACEHOLDER_ID, DID_ADD_SPECIAL = ensure_single_token_placeholder(tokenizer)
print("Placeholder:", PLACEHOLDER_STR, PLACEHOLDER_ID)

# 6) Oracle prompt ids (must match how you trained / ran inference)
ORACLE_PROMPT = (
    "You are an activation oracle. "
    "Using ONLY the injected activations, output the next text exactly.\n"
    "Next text:"
)
prompt_ids = tokenizer(ORACLE_PROMPT, add_special_tokens=False).input_ids
PROMPT_LEN = len(prompt_ids)
print("PROMPT_LEN:", PROMPT_LEN)

# 7) Inference stability: disable grad checkpointing, allow KV cache
try:
    model.gradient_checkpointing_disable()
except Exception:
    try:
        model.base_model.model.gradient_checkpointing_disable()
    except Exception:
        pass

try:
    model.config.use_cache = True
except Exception:
    pass

# 8) AO runtime state + hooks
@dataclass
class Injector:
    enabled: bool = False
    vecs: torch.Tensor = None  # [B,K,H]
    pos: torch.Tensor = None   # [B,K]

injector = Injector()
cache = {}

def layers(m):
    # Gemma + PEFT: PeftModel.base_model.model is GemmaForCausalLM; blocks at .model.layers
    return m.base_model.model.model.layers

def capture_hook(_, __, out):
    hs = out[0] if isinstance(out, (tuple, list)) else out
    if hs.dim() == 2:  # [S,H] -> [1,S,H]
        hs = hs.unsqueeze(0)
    cache["hs"] = hs.detach()

def inject_pre(_, inp):
    if not injector.enabled:
        return None

    hs = inp[0]
    squeezed = False
    if hs.dim() == 2:  # rare
        hs = hs.unsqueeze(0)
        squeezed = True

    v = injector.vecs.to(device=hs.device, dtype=hs.dtype)
    p = injector.pos.to(hs.device).long()

    B, S, H = hs.shape

    # IMPORTANT for generate(): on cached steps S can be 1; placeholders are not in this chunk
    if p.numel() == 0:
        return None
    pmin = int(p.min().item())
    pmax = int(p.max().item())
    if pmin < 0 or pmax >= S:
        return None

    sel = hs.gather(1, p.unsqueeze(-1).expand(-1, -1, H))  # [B,K,H]
    add = sel.norm(dim=-1, keepdim=True) * v / (v.norm(dim=-1, keepdim=True) + 1e-6)

    hs2 = hs.clone()
    b_idx = torch.arange(B, device=hs.device)
    K = p.size(1)
    for k in range(K):
        hs2[b_idx, p[:, k]] = hs2[b_idx, p[:, k]] + add[:, k]

    if squeezed:
        hs2 = hs2[0]
    return (hs2,) + inp[1:]

# Remove old hooks if cell is re-run
if "capture_handle" in globals() and capture_handle is not None:
    try: capture_handle.remove()
    except: pass
if "inject_handle" in globals() and inject_handle is not None:
    try: inject_handle.remove()
    except: pass

# Choose your layers (must match your notebook choices)
CAPTURE_LAYER = 9
INJECT_LAYER = 1

capture_handle = layers(model)[CAPTURE_LAYER].register_forward_hook(capture_hook)
inject_handle  = layers(model)[INJECT_LAYER].register_forward_pre_hook(inject_pre)

print("Hooks registered: capture", CAPTURE_LAYER, "| inject", INJECT_LAYER)

# 9) Step 1 helpers
@torch.no_grad()
def capture_activations(prompt: str):
    cache.clear()
    injector.enabled = False

    enc = tokenizer(prompt, return_tensors="pt")
    enc = {k: v.to(DEV) for k, v in enc.items()}

    # TARGET pass: disable adapter
    with model.disable_adapter():
        model(**enc, use_cache=False)

    hs = cache["hs"]
    if hs.dim() == 2:
        hs = hs.unsqueeze(0)

    return hs, enc["input_ids"]

@torch.no_grad()
def run_ao_from_activations(hs, act_positions, max_new_tokens=80):
    hs = hs.to(DEV)
    act_positions = act_positions.to(DEV).long()

    B, S, H = hs.shape
    assert 0 <= int(act_positions.min()) and int(act_positions.max()) < S

    # vectors to inject from target hs
    injector.vecs = hs.gather(1, act_positions.unsqueeze(-1).expand(-1, -1, H))

    # placeholder positions in oracle input
    K = act_positions.size(1)
    injector.pos = torch.arange(PROMPT_LEN, PROMPT_LEN + K, device=DEV).unsqueeze(0).expand(B, -1)
    injector.enabled = True

    oracle_input_ids = torch.tensor(
        prompt_ids + [PLACEHOLDER_ID]*K,
        dtype=torch.long,
        device=DEV
    ).unsqueeze(0).expand(B, -1)

    out = model.generate(
        oracle_input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    injector.enabled = False

    # Decode only the generated continuation
    gen = out[0, oracle_input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

print("AO runtime ready ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 47.4 MB/s eta 0:00:00
Mounted at /content/drive


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Loaded AO adapter from: /content/drive/MyDrive/gemma_ao_phaseA_lora
DEV: cuda:0
Placeholder: ? 235336
PROMPT_LEN: 22
Hooks registered: capture 9 | inject 1
AO runtime ready ✅


In [ ]:
prompt = "Alan Turing was a pioneering computer scientist who"
hs, input_ids = capture_activations(prompt)

S = input_ids.size(1)
K_ACT = 16
K = min(K_ACT, S)
act_positions = torch.arange(S-K, S, device=DEV).unsqueeze(0)

print(run_ao_from_activations(hs, act_positions))

 developed the theory of universal computation and the Turing test, which is a measure of a machine's ability to exhibit intelligent behaviour. Turing was also a key figure in the development of the computer,


## AmbiK fine-tune

In [ ]:
!pip -q install -U pandas
import pandas as pd, os, re
from sklearn.model_selection import train_test_split

!rm -rf AmbiK-dataset
!git clone -q https://github.com/cog-model/AmbiK-dataset

DATA_DIR = "AmbiK-dataset/ambik_dataset"
test_path  = os.path.join(DATA_DIR, "ambik_test_900.csv")

df = pd.read_csv(test_path)
print("Loaded:", df.shape)
print("Columns (raw):", list(df.columns)[:30])

def norm_cols(df):
    df = df.copy()
    df.columns = [re.sub(r"[^a-z0-9]+", "_", c.strip().lower()) for c in df.columns]
    return df

df = norm_cols(df)
print("Columns (norm):", list(df.columns)[:30])

def pick(df, *cands):
    for c in cands:
        if c in df.columns:
            return c
    raise KeyError(f"None of {cands} found. Columns: {df.columns.tolist()}")

C_ENV  = pick(df, "environment_short", "environment_full", "environment")
C_AMB  = pick(df, "ambiguous_task", "ambiguous")
C_UNA  = pick(df, "unambiguous_direct", "unambiguous_indirect", "unambiguous")
C_Q    = pick(df, "question", "clarifying_question")

# Split by original rows (keeps amb/unamb paired per environment)
SEED = 42
train_rows, dev_rows = train_test_split(df, test_size=0.2, random_state=SEED, shuffle=True)

print("Train rows:", train_rows.shape, "Dev rows:", dev_rows.shape)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 85.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.
cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.
Loaded: (900, 15)
Columns (raw): ['Unnamed: 0', 'environment_short', 'environment_full',

Column normalization + example builder

In [ ]:
NEG_LABEL = "NO_QUESTION"

def make_target_prompt(env, task):
    # Keep simple; you can add more structure later
    return f"Environment: {env}\nUser instruction: {task}\n"

def build_examples(rows_df):
    ex = []
    for _, r in rows_df.iterrows():
        env = str(r[C_ENV]).strip()
        amb = str(r[C_AMB]).strip()
        una = str(r[C_UNA]).strip()
        q   = str(r[C_Q]).strip()

        # ambiguous example
        ex.append({
            "target_text": make_target_prompt(env, amb),
            "label_text": q if q else "What would you like me to clarify?",
            "is_amb": True,
        })
        # unambiguous example
        ex.append({
            "target_text": make_target_prompt(env, una),
            "label_text": NEG_LABEL,
            "is_amb": False,
        })
    return ex

train_examples = build_examples(train_rows)
dev_examples   = build_examples(dev_rows)

print("Train examples:", len(train_examples), "Dev examples:", len(dev_examples))
print(train_examples[0])

Train examples: 1440 Dev examples: 360
{'target_text': 'Environment: knife block, porcelain cup, beer mug, ceramic mug, glass mug, plastic cup, paper cup, glass, ladle, glass milk bottle, oat milk bottle, cheesecake, chocolate cake, ice cream cake, kiwi, pear, stainless steel dinner fork, tangerine, banana, cutting board, apricot, watermelon, chopsticks, avocado, stainless steel salad fork\nUser instruction: Retrieve a cup from the cupboard.\n', 'label_text': 'Which type of cup should I retrieve from the cupboard?', 'is_amb': True}


Collator

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# Make sure these exist from your setup:
# - PLACEHOLDER_ID
# - K_ACT (e.g. 16)
# - capture/injection hooks already registered
# - model, tokenizer, injector, cache

AO_TASK_PROMPT = (
    "You are an activation oracle. Based ONLY on injected activations from the target model, "
    "decide if a clarifying question is needed.\n"
    f"If needed, output ONE clarifying question.\n"
    f"If not needed, output exactly: {NEG_LABEL}\n"
    "Output:"
)
ao_task_prompt_ids = tokenizer(AO_TASK_PROMPT, add_special_tokens=False).input_ids
AO_PROMPT_LEN = len(ao_task_prompt_ids)

class ExDataset(Dataset):
    def __init__(self, ex_list): self.ex = ex_list
    def __len__(self): return len(self.ex)
    def __getitem__(self, i): return self.ex[i]

def collate_phaseB(batch):
    b = batch[0]

    target_ids = tokenizer(b["target_text"], add_special_tokens=False).input_ids
    target_ids = target_ids[-512:]  # keep stable memory
    target = torch.tensor(target_ids, dtype=torch.long)

    S = target.numel()
    K = min(K_ACT, S)
    act_pos = torch.arange(S - K, S, dtype=torch.long)

    label_ids = tokenizer(b["label_text"], add_special_tokens=False).input_ids + [tokenizer.eos_token_id]

    oracle_ids = torch.tensor(
        ao_task_prompt_ids + [PLACEHOLDER_ID]*K + label_ids,
        dtype=torch.long
    )

    labels = torch.full_like(oracle_ids, -100)
    start = AO_PROMPT_LEN + K
    labels[start:] = oracle_ids[start:]

    return {
        "target_input_ids": target.unsqueeze(0),
        "target_attention_mask": torch.ones_like(target).unsqueeze(0),
        "act_positions": act_pos.unsqueeze(0),

        "oracle_input_ids": oracle_ids.unsqueeze(0),
        "oracle_attention_mask": torch.ones_like(oracle_ids).unsqueeze(0),
        "oracle_labels": labels.unsqueeze(0),

        "placeholder_positions": torch.arange(AO_PROMPT_LEN, AO_PROMPT_LEN + K).unsqueeze(0),
        "is_amb": torch.tensor([1 if b["is_amb"] else 0], dtype=torch.long),
        "gold_text": b["label_text"],
    }

train_dl = DataLoader(ExDataset(train_examples), batch_size=1, shuffle=True,  collate_fn=collate_phaseB)
dev_dl   = DataLoader(ExDataset(dev_examples),   batch_size=1, shuffle=False, collate_fn=collate_phaseB)

print("Dataloaders ready:", len(train_examples), len(dev_examples))

Dataloaders ready: 1440 360


Training loop

In [ ]:
from accelerate import Accelerator
from torch.optim import AdamW
from tqdm.auto import tqdm

PHASEB_EPOCHS = 8
PHASEB_LR = 1e-4
GRAD_ACCUM = 16
K_ACT = 16

acc = Accelerator(mixed_precision="fp16" if torch.cuda.is_available() else "no")
opt = AdamW(model.parameters(), lr=PHASEB_LR)

model.train()
model, opt, train_dl = acc.prepare(model, opt, train_dl)

for epoch in range(PHASEB_EPOCHS):
    pbar = tqdm(train_dl, desc=f"Phase B epoch {epoch+1}/{PHASEB_EPOCHS}")
    running = 0.0
    step = 0

    for b in pbar:
        # TARGET pass (LoRA disabled) to capture activations
        cache.clear()
        injector.enabled = False
        with torch.no_grad():
            with model.disable_adapter():
                model(
                    b["target_input_ids"],
                    b["target_attention_mask"],
                    use_cache=False
                )

        hs = cache["hs"]
        if hs.dim() == 2:
            hs = hs.unsqueeze(0)

        # Extract vectors from target, inject into oracle placeholders
        injector.vecs = hs.gather(
            1, b["act_positions"].unsqueeze(-1).expand(-1, -1, hs.size(-1))
        )
        injector.pos = b["placeholder_positions"]
        injector.enabled = True

        # ORACLE pass (LoRA enabled) supervised on label_text
        out = model(
            b["oracle_input_ids"],
            b["oracle_attention_mask"],
            labels=b["oracle_labels"],
            use_cache=False
        )

        loss = out.loss
        acc.backward(loss / GRAD_ACCUM)
        injector.enabled = False

        if (step + 1) % GRAD_ACCUM == 0:
            opt.step()
            opt.zero_grad(set_to_none=True)

        running += acc.gather(loss.detach()).mean().item()
        if step % 50 == 0:
            pbar.set_postfix(loss=float(loss.detach().cpu()))
        step += 1

print("Phase B training done ✅")

Phase B epoch 1/8:   0%|          | 0/1440 [00:00<?, ?it/s]

Phase B epoch 2/8:   0%|          | 0/1440 [00:00<?, ?it/s]

Phase B epoch 3/8:   0%|          | 0/1440 [00:00<?, ?it/s]

Phase B epoch 4/8:   0%|          | 0/1440 [00:00<?, ?it/s]

Phase B epoch 5/8:   0%|          | 0/1440 [00:00<?, ?it/s]

Phase B epoch 6/8:   0%|          | 0/1440 [00:00<?, ?it/s]

Phase B epoch 7/8:   0%|          | 0/1440 [00:00<?, ?it/s]

Dev evaluation

In [ ]:
import re
import torch
from tqdm.auto import tqdm

def normalize(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

@torch.no_grad()
def ao_predict_from_batch(b, max_new_tokens=64):
    # (re)capture target activations
    cache.clear()
    injector.enabled = False
    with model.disable_adapter():
        model(b["target_input_ids"], b["target_attention_mask"], use_cache=False)

    hs = cache["hs"]
    if hs.dim() == 2:
        hs = hs.unsqueeze(0)

    injector.vecs = hs.gather(
        1, b["act_positions"].unsqueeze(-1).expand(-1, -1, hs.size(-1))
    )
    injector.pos = b["placeholder_positions"]
    injector.enabled = True

    # Oracle prompt only (no labels)
    K = b["placeholder_positions"].size(1)
    oracle_input_ids = torch.tensor(
        ao_task_prompt_ids + [PLACEHOLDER_ID]*K,
        dtype=torch.long,
        device=b["oracle_input_ids"].device
    ).unsqueeze(0)

    out = model.generate(oracle_input_ids, max_new_tokens=max_new_tokens, do_sample=False)
    injector.enabled = False

    gen = out[0, oracle_input_ids.shape[1]:]
    return normalize(tokenizer.decode(gen, skip_special_tokens=True))

# Run eval
model.eval()
dev_dl_prepared = acc.prepare(dev_dl)

correct = 0
total = 0
examples_to_show = 8
shown = 0

for b in tqdm(dev_dl_prepared, desc="Dev eval"):
    pred = ao_predict_from_batch(b)
    pred_is_amb = (pred != NEG_LABEL)

    gold_is_amb = bool(b["is_amb"][0].item())
    correct += int(pred_is_amb == gold_is_amb)
    total += 1

    if shown < examples_to_show:
        print("\n---")
        print("GOLD TEXT:", b["gold_text"][0] if isinstance(b["gold_text"], (list, tuple)) else b["gold_text"])
        print("PRED     :", pred)
        print("GOLD ambiguous?", gold_is_amb, "| PRED ambiguous?", pred_is_amb)
        shown += 1

print("\nAmbiguity-detection accuracy:", correct/total)

Dev eval:   0%|          | 0/360 [00:00<?, ?it/s]


---
GOLD TEXT: Which type of container should I use to store the remaining blue cheese?
PRED     : Which type of container should be used to store the blue cheese?
GOLD ambiguous? True | PRED ambiguous? True

---
GOLD TEXT: NO_QUESTION
PRED     : NO_QUESTION
GOLD ambiguous? False | PRED ambiguous? False

---
GOLD TEXT: When you say 'stir until hot', do you mean I should stir until the chocolate dissolves, or should I reheat the coffee while stirring if it has cooled down?
PRED     : NO_QUESTION
GOLD ambiguous? True | PRED ambiguous? False

---
GOLD TEXT: NO_QUESTION
PRED     : NO_QUESTION
GOLD ambiguous? False | PRED ambiguous? False

---
GOLD TEXT: Which type of yogurt should be used for mixing with the zest?
PRED     : NO_QUESTION
GOLD ambiguous? True | PRED ambiguous? False

---
GOLD TEXT: NO_QUESTION
PRED     : NO_QUESTION
GOLD ambiguous? False | PRED ambiguous? False

---
GOLD TEXT: Which two additional types of fruit would you like me to use for your salad?
PRED     : Which othe

Save adapter

In [ ]:
SAVE_DIR = "gemma_ao_lora_phaseAplusB_ambik_split900"
acc.unwrap_model(model).save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved adapter+tokenizer to:", SAVE_DIR)

Saved adapter+tokenizer to: gemma_ao_lora_phaseAplusB_ambik_split900


Load adapter